## RAG with Vector Search

In [1]:
import os
from dotenv import load_dotenv
load_dotenv(dotenv_path="../../01_module_agentic_rag/.env")

from openai import OpenAI
openai_client = OpenAI(
    api_key=os.getenv("LMSTUDIO_API_KEY"),
    base_url=os.getenv("LMSTUDIO_HOST")
)

In [2]:
from sentence_transformers import SentenceTransformer

embed_model = SentenceTransformer("all-MiniLM-L6-v2")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [3]:
# Loading data source
from ingest import load_faq_data, build_index

documents = load_faq_data()
index = build_index(documents)

In [4]:
# Preparing data - embedding both question + answer together
# question and answer will be embedded both together so that a query can match against both the question text and answer text in the index.
texts = []

for doc in documents:
    text = doc["question"] + " " + doc["answer"]
    texts.append(text)

In [5]:
# Next step is to generate embeddings
# We need to embed in batches as processing can take a long time
# import tqdm to review process
from tqdm.auto import tqdm

In [7]:
# Now chunk the dataset into batches and encode each batch
batch_size = 50
vectors = []

for i in tqdm(range(0, len(texts), batch_size)):
    batch = texts[i:i + batch_size]
    batch_vectors = embed_model.encode(batch)
    vectors.extend(batch_vectors)

len(vectors)

  0%|          | 0/27 [00:00<?, ?it/s]

1350

In [8]:
# Next we turn it into a 2-D array (matrix) where row = docs and cols = dimension of the vector which represent clues or meanings of the text the model has learned and is represented by a number
import numpy as np
X = np.array(vectors)
X

array([[-0.02670616, -0.12245759,  0.01594421, ..., -0.0023065 ,
        -0.11218402, -0.02365559],
       [-0.01099559, -0.11074748, -0.02536943, ...,  0.09022236,
        -0.02697359,  0.01975664],
       [-0.08896557, -0.06128179,  0.00775611, ...,  0.04059712,
         0.00479282, -0.02745942],
       ...,
       [-0.03652923,  0.0141543 , -0.06838647, ...,  0.04316789,
         0.08105531, -0.02148631],
       [-0.1309159 , -0.06990598, -0.0093188 , ..., -0.00044342,
        -0.01285736,  0.01426917],
       [-0.07984782,  0.01926984,  0.02544974, ..., -0.03368021,
        -0.01884026,  0.05837055]], shape=(1350, 384), dtype=float32)

In [9]:
from rag_helper import RAGBase, LMStudioRAG, RAGVector
model="qwen/qwen3.5-9b"

In [10]:
assistant = LMStudioRAG(index, openai_client, model=model)

In [11]:
query = "I just found this about this program, can I still sign up?"
assistant.rag(query)

'Yes, you can still sign up! However, please note that if your goal is to receive a certificate, you must submit your project while the program is still accepting submissions. Homework is not mandatory for a certificate (only the Capstone project is required), but it is recommended for learning and affects your leaderboard rank.'

In the above example, we are still utilzing key word text search. Next we will replace it with vector search.

In [13]:
# First we need the documents that have been converted into a vector and indexed to pass on to the assistant
from minsearch import VectorSearch

vindex = VectorSearch(keyword_fields=["course"])
vindex.fit(X, documents)

In [14]:
# Now we initialize our RAG with vector search
vector_assistant = RAGVector(
    embedder=embed_model,
    index=vindex,
    llm_client=openai_client,
)

In [16]:
vector_assistant.rag("Oh no! Did I miss the course? I want to sign up really bad!")

"No, you haven't missed the course! You can still join.\n\nAccording to the provided context:\n*   **Registration:** You don't actually need to register officially (you are already accepted). While the registration form is open, you can start learning and submit your homework immediately.\n*   **Certificate:** If you plan to receive a certificate later, you just need to make sure you submit your project while submissions are still being accepted.\n\nYou can go ahead and start learning right away!"